# WESAD: 전통적 ML + 설명 가능한 휴식 알림

실제 실행 결과는 `RESULTS_KO.md`를 참조하세요. 이 노트북은 실행 순서 안내입니다. 저장된 셀 출력은 없으며 사용자 실험은 수행 전입니다. 프로토콜의 고정 설정을 바꾸면 별도 실험으로 기록하세요.

In [ ]:
from pathlib import Path
import json, subprocess, sys
ROOT = Path.cwd()
assert (ROOT / "experiment.py").exists(), "압축 파일을 풀고 프로젝트 폴더에서 실행하세요"


## 1. 데이터 및 특징 추출
공식 원본 파일만 사용하세요. 다운로드 약 2.25GB. 설치는 README를 따르세요. 이미 포함된 features.csv로 모델을 재학습할 수도 있습니다.

In [ ]:
# 원본을 내려받은 후 필요할 때만 실행
# subprocess.run([sys.executable,"experiment.py","extract","--zip","data/WESAD.zip"], check=True)
import pandas as pd
f = pd.read_csv("results/features.csv")
f.groupby(["subject", "condition"]).size().unstack(fill_value=0)

## 2. 참가자별 LOSO
모든 전처리를 fold 안에서 학습합니다. 고정 설정의 기준선/로지스틱/RF를 비교합니다.

In [ ]:
# 모델과 설명을 다시 생성할 때 아래 줄의 주석 해제
# subprocess.run([sys.executable,"experiment.py","train"], check=True)
scores = pd.read_csv("results/fold_metrics.csv")
scores.groupby("model")[["macro_f1","balanced_accuracy","stress_recall","roc_auc"]].mean()

## 3. 동일한 RF의 out-of-fold 예측 설명
SHAP의 전체 기여도 합을 검증하고 LIME 국소 R², Anchors precision/coverage를 기록합니다. 설명 생성에는 fold 모델이 필요하므로 압축 파일에서 재실행하려면 train부터 실행하세요.

In [ ]:
# subprocess.run([sys.executable,"experiment.py","explain"], check=True)
cases=json.loads(Path("results/explanations.json").read_text())
cases[0]

## 4. HCI 수업 파일럿
프로토콜을 읽고 16명의 배정표를 사용하세요. 정답 키는 참가자에게 공개하지 않습니다. 빈 응답은 결과가 아닙니다.

In [ ]:
schedule = pd.read_csv("results/hci_response_template.csv")
schedule.head()

In [ ]:
# 실제 응답을 수집한 뒤 실행
# subprocess.run([sys.executable,"analyze_hci.py","collected_responses.csv"],check=True)

## 해석 범위
실험 조건을 분류한 것이며 개인별 스트레스 진단·생산성 향상·휴식 효과 검증이 아닙니다. 원논문과 다른 특징/창 설정의 수업용 독립 실험입니다.